## TIFF plot

In [1]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")

# install.packages("ncdf4", "tidyverse", "terra", "dplyr", "sf", "jsonlite", "utils")
# install.packages("ncdf4")

# GET THE DEFAULT INITIAL PARAMETERS

## CONDA ENVIRONEMENT (OPTIONAL)
Sys.getenv("CONDA_DEFAULT_ENV")
## VERSION OF R IN USE
version
# getwd()
# list.files("data/2008")

# setwd("/path/to/your/folder")


system("conda activate")

[1] "base"

               _                           
platform       x86_64-conda-linux-gnu      
arch           x86_64                      
os             linux-gnu                   
system         x86_64, linux-gnu           
status                                     
major          4                           
minor          5.2                         
year           2025                        
month          10                          
day            31                          
svn rev        88974                       
language       R                           
version.string R version 4.5.2 (2025-10-31)
nickname       [Not] Part in a Rumble      

In [2]:
library(ncdf4)
library(R.utils)
library(tidyverse)
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)
# ---
library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings


Warning message:
“package ‘ggplot2’ was built under R version 4.5.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr   

In [1]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"data",sep = "/")
outputDir <- paste(workdir,"outputs",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

# NASA Data - SST

In [32]:
# 2.2 DOWNLOAD NASA NetCDF ----
NASA.cmd <- paste("python", paste(scriptDir,"NASA_API.py",sep="/") )
system(NASA.cmd)

In [33]:
# 2.2 ANALYSIS ----
SST_file <- paste(dataDir, "2008/AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.nc", sep= "/")
file.exists(SST_file)

[1] FALSE

In [34]:
# Open the NetCDF file

nc_sst_file <- nc_open(SST_file)
print(nc_sst_file)

Error in R_nc4_open: No such file or directory


ERROR: Error in nc_open(SST_file): Error in nc_open trying to open file data/2008/AQUA_MODIS.20071201_20071231.L3m.MO.SST.sst.9km.nc (return_on_error= FALSE )


Based on the time coverage shown above:
```
...
    61 global attributes:
        product_name: AQUA_MODIS.20071101_20071130.L3m.MO.SST.sst.9km.nc
        instrument: MODIS
        ...
        time_coverage_start: 2007-11-01T00:10:01.000Z
        time_coverage_end: 2007-12-01T02:05:00.000Z
        ...
...
```
we can see that the time coverage of the satellite extends until the first hours of the next month. Hence, the reason why the API included november monthly measures. 

In [ ]:
# Structure of the file

names(nc_sst_file)
# Names of the variables
names(nc_sst_file$var)
# Names of the dimensions
names(nc_sst_file$dim)

In [ ]:
#  Extract the coordinates
dim_sst_lon <- ncvar_get(nc_sst_file, "lon")
dim_sst_lat <- ncvar_get(nc_sst_file, "lat")
dim_sst <- ncvar_get(nc_sst_file, "sst")

names(nc_sst_file$var$sst)

In [ ]:
dim_sst_lon[1:5]
dim_sst_lat[1:5]
dim_sst[1:5]

In [ ]:
# summary(dim_sst)
sum(is.na(dim_sst))
all(is.na(dim_sst))
dim_sst <- ncvar_get(nc_sst_file, "sst", raw_datavals = TRUE)
range(dim_sst)

In [ ]:
sst_raw <- ncvar_get(nc_sst_file, "sst", raw_datavals = TRUE)

# Get attributes
fill_value <- ncatt_get(nc_sst_file, "sst", "_FillValue")$value ; fill_value
scale      <- ncatt_get(nc_sst_file, "sst", "scale_factor")$value ; scale

sst_raw[sst_raw == fill_value] <- NA
sst <- sst_raw * scale
# Fill Missing values by replacing NA with the lowest values
range(dim_sst, na.rm = TRUE)

In [ ]:
sst_coords <- expand.grid(dim_sst_lon, dim_sst_lat)
sst_matrix <- cbind(sst_coords, as.vector(dim_sst))
names(sst_matrix) <- c("lon", "lat", "sst")
dim_sst[1:5]
dim_sst[1:5]
head(sst_matrix)

In [ ]:
# Close file
nc_close(nc_sst_file)

---

In [3]:
global_topo_tiff_gz <- "global_topo.tiff.gz"

# nchar(global_topo_tiff)
filename.length <- nchar(global_topo_tiff_gz)

# Get the extension ("." + 2 letters)
substr(global_topo_tiff_gz,filename.length-2, filename.length)

# Get the name without the extension ("." + 2 letters)
start <- 1
end <- filename.length-3
global_topo_tiff <- substr(global_topo_tiff_gz,start, end)

print(global_topo_tiff_gz)
print(global_topo_tiff)

# ==============================================================================

if(!file.exists(global_topo_tiff_gz)){
    download.file("https://topex.ucsd.edu/pub/global_topo_tiff/global.tiff.gz", global_topo_tiff_gz, mode = "wb")
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================

if(!file.exists(global_topo_tiff)){
    R.utils::gunzip(global_topo_tiff_gz, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
}


[1] ".gz"

[1] "global_topo.tiff.gz"
[1] "global_topo.tiff"


In [4]:
# ==============================================================================
# FILE INFO
# ==============================================================================
print(file.info( substr(global_topo_tiff_gz,start, end) ))
print(file.info( substr(global_topo_tiff,start, end) ))

ls()

                     size isdir mode               mtime               ctime
global_topo.tiff 27104274 FALSE  644 2026-05-19 07:26:54 2026-05-19 07:26:54
                               atime uid gid  uname grname
global_topo.tiff 2026-05-19 07:26:54 999 100 jovyan  users
                     size isdir mode               mtime               ctime
global_topo.tiff 27104274 FALSE  644 2026-05-19 07:26:54 2026-05-19 07:26:54
                               atime uid gid  uname grname
global_topo.tiff 2026-05-19 07:26:54 999 100 jovyan  users


[1] "end"                 "filename.length"     "global_topo_tiff"   
[4] "global_topo_tiff_gz" "start"

In [5]:
# MOVE TO DATA DIRECTORY 
data.directory <- "data"
ifelse(!dir.exists(file.path(data.directory)),
        dir.create(file.path(data.directory)),
        "Directory Exists")

file.rename(from=global_topo_tiff_gz,
            to=paste(data.directory, global_topo_tiff_gz, sep = "/"))
file.rename(from=global_topo_tiff,
            to=paste(data.directory, global_topo_tiff, sep = "/"))

[1] TRUE

Warning message in file.rename(from = global_topo_tiff_gz, to = paste(data.directory, :
“cannot rename file 'global_topo.tiff.gz' to 'data/global_topo.tiff.gz', reason 'No such file or directory'”


[1] FALSE

[1] TRUE

## NetCDF (Copernicus Data Files)

In [6]:
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")

In [7]:
library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics
library(dplyr)

In [8]:
ls()

[1] "data.directory"      "end"                 "filename.length"    
[4] "global_topo_tiff"    "global_topo_tiff_gz" "start"

In [9]:
# 2.2 DOWNLOAD AND LOAD BATHYMETRY ----
options(timeout = 600)  # 10 minutes
bathy_file <- "global_topo_1min_topo_19_1.nc"
if(file.exists(bathy_file)){
    cat(bathy_file, "is (are) already in your repertory.")
    } else {
    download.file("https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc", bathy_file, mode = "wb")
    print('File Downloaded')
}


[1] "File Downloaded"


In [10]:
# MOVE TO DATA DIRECTORY 
data.directory <- "data"
ifelse(!dir.exists(file.path(data.directory)),
        dir.create(file.path(data.directory)),
        "Directory Exists")

file.rename(from=bathy_file,
            to=paste(data.directory, bathy_file, sep = "/"))
bathy_file <- paste(data.directory, bathy_file, sep = "/")

[1] "Directory Exists"

[1] TRUE

## Data Exploration

File exploration : ornldaac [github repository](https://github.com/ornldaac/netCDF_data_in_R/blob/master/netCDF_in_r_ornldaac_tutorial.md)

In [11]:
## A.7 Draw Pairs Plot of Data Frame Columns
'install.packages("GGally")             # Install GGally package
library("GGally")                      # Load GGally package'

# ggpairs(Bathy)                        # Draw pairs plot

[1] "install.packages(\"GGally\")             # Install GGally package\nlibrary(\"GGally\")                      # Load GGally package"

In [12]:
## A.8 Boxplots of Multiple Columns 
# ggplot(as.data.frame(Bathy),                    # Draw boxplots
#        aes(x = value,
#            fill = name)) +
#   geom_boxplot()

In [13]:
## A.9 Histograms of Multiple Columns 
# ggplot(Bathy,                    # Draw histograms
#        aes(x = value)) +
#   geom_histogram() + 
#   facet_wrap(name ~ ., scales = "free")

In [14]:
cat("Content of", getwd(), ":\n", list.files(), "\n")
cat(bathy_file,"exists:", file.exists(bathy_file),"\n")
# Check for the file
cat("Size:", file.info(bathy_file)$size)
 # If size is 0 or very small, the file is broken.

Content of /data/jwd07/pulsar_staging/102738602/working/jupyter :
 API_Copernicus.py API_Copernicus.sh APIs.py CLARA_clustering.r data Data Executed_JupyTool_Notebook.ipynb galaxy_inputs Normalization.r outputs process_in3.ipynb Test-PELAGIC-1.ipynb UPGMA_clustering.r 
data/global_topo_1min_topo_19_1.nc exists: TRUE 
Size: 548098492

In [15]:
# Open the NetCDF file

nc_file <- nc_open(bathy_file)
print(nc_file)

# Structure of the file

names(nc_file)
# Names of the variables
names(nc_file$var)
# Names of the dimensions
names(nc_file$dim)

File data/global_topo_1min_topo_19_1.nc (NC_FORMAT_NETCDF4):

     1 variables (excluding dimension variables):
        float z[lon,lat]   (Chunking: [129,128])  (Compression: shuffle,level 3)
            long_name: z
            _FillValue: NaN
            actual_range: -10926.9326171875
             actual_range: 8516.9990234375

     2 dimensions:
        lon  Size:21600 
            long_name: longitude
            units: degrees_east
            actual_range: -180
             actual_range: 180
        lat  Size:9600 
            long_name: latitude
            units: degrees_north
            actual_range: -80
             actual_range: 80

    6 global attributes:
        Conventions: COARDS, CF-1.5
        title: 
        history: grdsample -R-180/180/-80/80 -I1m @GMTAPI@-000001 -Gtopo_19.1.nc -fg --GMT_HISTORY=false
        description: 
        GMT_version: 5.4.5 [64-bit]
        node_offset: 1


[1] "filename"    "writable"    "id"          "error"       "safemode"   
 [6] "format"      "is_GMT"      "groups"      "fqgn2Rindex" "ndims"      
[11] "natts"       "dim"         "unlimdimid"  "nvars"       "var"

[1] "z"

[1] "lon" "lat"

In [16]:
# List the attributes and sub-attributes

i <- 1
for(listVar in names(nc_file)){
    cat(i, listVar,"\n")
    for(listNames in names(nc_file[[listVar]])){
        cat("Attr:", listNames, ":", names(nc_file[[listVar]][[listNames]]),"\n")
    }
    i <- i + 1
}

1 filename 
2 writable 
3 id 
4 error 
5 safemode 
6 format 
7 is_GMT 
8 groups 
9 fqgn2Rindex 
Attr:  : 
10 ndims 
11 natts 
12 dim 
Attr: lon : name len unlim group_index group_id id dimvarid units vals create_dimvar 
Attr: lat : name len unlim group_index group_id id dimvarid units vals create_dimvar 
13 unlimdimid 
14 nvars 
15 var 
Attr: z : id name ndims natts size dimids prec units longname group_index chunksizes storage shuffle compression dims dim varsize unlim make_missing_value missval hasAddOffset hasScaleFact 


In [17]:
# Sub-Content for lat and lon
names(nc_file$dim$lon)
names(nc_file$dim$lat)

nc_file$dim$lon[1:5]
nc_file$dim$lat[1:5]

# 5 first Values
nc_file$dim$lon$vals[1:5] # print(nc_file$dim['lon'])
nc_file$dim$lat$vals[1:5]
nc_file$var$z$id[1:5]


[1] "name"          "len"           "unlim"         "group_index"  
 [5] "group_id"      "id"            "dimvarid"      "units"        
 [9] "vals"          "create_dimvar"

[1] "name"          "len"           "unlim"         "group_index"  
 [5] "group_id"      "id"            "dimvarid"      "units"        
 [9] "vals"          "create_dimvar"

$name
[1] "lon"

$len
[1] 21600

$unlim
[1] FALSE

$group_index
[1] 1

$group_id
[1] 65536

$name
[1] "lat"

$len
[1] 9600

$unlim
[1] FALSE

$group_index
[1] 1

$group_id
[1] 65536

[1] -179.9917 -179.9750 -179.9583 -179.9417 -179.9250

[1] -79.99167 -79.97500 -79.95833 -79.94167 -79.92500

$id
[1] 2

$group_index
[1] -1

$group_id
[1] 65536

$list_index
[1] 1

$isdimvar
[1] FALSE

In [18]:
# Get coordinates variables
longitude <- ncvar_get(nc_file,"lon")
latitude <- ncvar_get(nc_file,"lat")
z <- ncvar_get(nc_file,"z")

In [19]:
# Dimensions of latitude & longitude
print(c(length(longitude), length(latitude)))
print(dim(z))

# Check longtitude and latitude values
cat(" Head of longitude:",head(longitude),"\n")
cat(" Head of latitude:",head(latitude),"\n")

fillvalue <- ncatt_get(nc_file, "z", "_FillValue") # The fill value (aka, the no data value) is -9999.
print(fillvalue$value)

[1] 21600  9600
[1] 21600  9600
 Head of longitude: -179.9917 -179.975 -179.9583 -179.9417 -179.925 -179.9083 
 Head of latitude: -79.99167 -79.975 -79.95833 -79.94167 -79.925 -79.90833 
[1] NaN


In [20]:
nc_close(nc_file)

In [21]:
z[z == fillvalue$value] <- NA
nc.slice.min80 <- z[,1]
dim(nc.slice.min80)

NULL

In [22]:
#r <- rast(nc.slice.min80,
#  extent = ext(min(lon), max(lon), min(lat), max(lat)),
#  crs = "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs +towgs84=0,0,0"
#)
#rm(nc.slice.min80)

### Plots

In [23]:
Bathy <- rast(bathy_file)
print("Converted into Raster File")

Warning message:
“[rast] unknown extent”


[1] "Converted into Raster File"


In [24]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/Fig1.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()


agg_record_1928897348 
                    2

### Convert NetCDF to CSV
Based on [Copernicus Marine Services](https://help.marine.copernicus.eu/en/articles/6328012-how-to-convert-netcdf-to-csv-using-r)

In [25]:
#  Extract the coordinates
nc_file <- nc_open(bathy_file)

dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)

In [26]:
dim_lon[1:5]
dim_lat[1:5]
dim_depth[1:5]

[1] -179.9917 -179.9750 -179.9583 -179.9417 -179.9250

[1] -79.99167 -79.97500 -79.95833 -79.94167 -79.92500

[1] -150.0287 -150.0287 -150.1319 -150.1033 -149.9649

In [27]:

check_dim <- function(dim_var){
    cat(str(dim_var),
    length(dim_var),
    any(is.na(dim_var)),sep="\n")
}

check_dim(dim_lon)
print("---*---")
check_dim(dim_lat)
print("---*---")
check_dim(dim_depth)

 num [1:21600(1d)] -180 -180 -180 -180 -180 ...

21600
FALSE
[1] "---*---"
 num [1:9600(1d)] -80 -80 -80 -79.9 -79.9 ...

9600
FALSE
[1] "---*---"
 num [1:21600, 1:9600] -150 -150 -150 -150 -150 ...

207360000
FALSE


In [28]:
dim(dim_depth)

[1] 21600  9600

In [29]:
coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))

In [30]:
names(depth_matrix) <- c("lon", "lat", "depth")
head(depth_matrix)
# head(df.depth)
nc_close(nc_file)

,lon,lat,depth
,<dbl[1d]>,<dbl[1d]>,<dbl>
1,-179.9917,-79.99167,-150.0287
2,-179.9750,-79.99167,-150.0287
3,-179.9583,-79.99167,-150.1319
4,-179.9417,-79.99167,-150.1033
5,-179.9250,-79.99167,-149.9649
6,-179.9083,-79.99167,-149.9363


In [31]:
output.directory <- "outputs"
ifelse(!dir.exists(file.path(output.directory)),
        dir.create(file.path(output.directory)),
        "Directory Exists")

head(na.omit(depth_matrix), 5)  # Display some non-NaN values for a visual check
csv_fname <- paste(output.directory,"netcdf_depth.csv", sep="/")
write.table(depth_matrix, csv_fname, row.names=FALSE, sep=";")

[1] "Directory Exists"

,lon,lat,depth
,<dbl[1d]>,<dbl[1d]>,<dbl>
1,-179.9917,-79.99167,-150.0287
2,-179.9750,-79.99167,-150.0287
3,-179.9583,-79.99167,-150.1319
4,-179.9417,-79.99167,-150.1033
5,-179.9250,-79.99167,-149.9649
